In [ ]:
import os, json, re, gc, torch, pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from tqdm.auto import tqdm

In [ ]:
# ===== Config =====
MODEL_NAME = '/root/autodl-tmp/employee_code/base_model'
EVAL_JSON  = '/root/autodl-tmp/cpl_ft/data/eval_cpl_questions.json'
CKPT_DIR   = '/root/autodl-tmp/cpl_ft/checkpoints'
FINE_DIR   = '/root/autodl-tmp/cpl_ft/fine_checkpoints'
RESULTS_DIR = '/root/autodl-tmp/cpl_ft/eval_results'
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
# ===== Tokenizer =====
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print('Tokenizer loaded')

In [ ]:
# ===== Eval constants =====
TOP5_PROMPT = '你是一个专业的中国法律检索专家。\n\n你的任务不是作答，而是为后续数据库检索生成候选法条。\n请根据用户案情，输出最相关的5条法条编号，按相关性从高到低排序。\n\n硬性要求：\n1. 必须恰好输出5条；\n2. 每条都必须是《法律名称》第XXX条；\n3. 不要输出解释、分析、理由、序号或其他任何文字；\n4. 即使不确定，也必须给出5条最可能的候选法条；\n5. 优先输出最核心、最直接适用的法条；\n6. 只输出 JSON，不要输出其他内容。\n\n输出格式：\n{"articles": ["《法律名称》第XXX条", ...]}'

ART_RE = r'《.+?》第[0-9０-９一二三四五六七八九十百千万零〇○两]+条'

CN_M = {'零':0,'〇':0,'○':0,'一':1,'二':2,'两':2,'三':3,'四':4,'五':5,'六':6,'七':7,'八':8,'九':9}
CN_U = {'十':10,'百':100,'千':1000,'万':10000}

def c2i(t):
    if not t: return None
    t = str(t).strip().translate(str.maketrans('０１２３４５６７８９','0123456789'))
    if t.isdigit(): return int(t)
    total = section = number = 0
    for ch in t:
        if ch in CN_M: number = CN_M[ch]
        elif ch in CN_U:
            u = CN_U[ch]
            if u == 10000:
                section = (section + (number or 0)) * u
                total += section
                section = number = 0
            else:
                if number == 0: number = 1
                section += number * u
                number = 0
        else: return None
    return total + section + number

def canon(art):
    if not isinstance(art, str): return ''
    x = art.strip()
    x = re.sub(r'\s+', '', x)
    x = x.replace('（', '(').replace('）', ')')
    x = x.translate(str.maketrans('０１２３４５６７８９', '0123456789'))
    m = re.match(r'^(《.*?》)第([0-9一二三四五六七八九十百千万零〇○两]+)条$', x)
    if not m: return x
    law = re.sub(r'^《中华人民共和国', '《', m.group(1))
    no = c2i(m.group(2))
    return f'{law}第{no}条' if no is not None else f'{law}第{m.group(2)}条'

def extract_articles(text, topk=5):
    mentions = []
    m = re.search(r'\{.*\}', text, re.S)
    if m:
        try:
            obj = json.loads(m.group())
            if isinstance(obj, dict) and isinstance(obj.get('articles'), list):
                for x in obj['articles']:
                    if isinstance(x, str):
                        mentions.extend(re.findall(ART_RE, x))
        except: pass
    if not mentions:
        mentions = re.findall(ART_RE, text)[:topk]
    else:
        for m2 in re.findall(ART_RE, text):
            if len(mentions) >= topk: break
            mentions.append(m2)
    return [canon(x) for x in mentions[:topk] if canon(x)]

@torch.no_grad()
def generate_top5(question, model):
    msgs = [
        {'role': 'system', 'content': TOP5_PROMPT},
        {'role': 'user', 'content': f'/no_think\n案情：\n{question}\n\n请严格输出 JSON。'}
    ]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=256, do_sample=True, repetition_penalty=1.05,
                              eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id)
    raw = tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True).strip()
    return {'raw': raw, 'pred': extract_articles(raw)}

In [ ]:
# ===== Evaluate one adapter =====
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32)

with open(EVAL_JSON, 'r', encoding='utf-8') as f:
    eval_data = json.load(f)
print(f'Eval questions: {len(eval_data)}')

# Collect checkpoints from BOTH directories
targets = []
for src_label, src_dir in [('orig', CKPT_DIR), ('fine', FINE_DIR)]:
    if not os.path.isdir(src_dir):
        print(f'SKIP: {src_dir} not found')
        continue
    for name in sorted(os.listdir(src_dir)):
        full = os.path.join(src_dir, name)
        if os.path.isdir(full) and name.startswith('checkpoint-'):
            s = name.split('-')[-1]
            if s.isdigit():
                targets.append((int(s), f'{name}({src_label})', full))
    final = os.path.join(src_dir, 'final_adapter')
    if os.path.isdir(final):
        tag = 9999 if src_label == 'fine' else 9998
        targets.append((tag, f'final_adapter({src_label})', final))

targets = sorted(targets, key=lambda x: x[0])
print(f'Total checkpoints: {len(targets)}')
for step, name, path in targets:
    print(f'  {name}  ->  {path}')

In [ ]:
# ===== Evaluate all + Leaderboard =====
all_rows = []

for step, label, adapter_path in targets:
    print(f'\n{"="*60}')
    print(f'Evaluating: {label}')
    print(f'{"="*60}')

    gc.collect()
    torch.cuda.empty_cache()

    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map='auto', trust_remote_code=True)
    model = PeftModel.from_pretrained(base_model, adapter_path, is_trainable=False)
    model.eval()

    results = []
    tp_total = pred_total = gt_total = 0

    for item in tqdm(eval_data, desc=f'Running {label}'):
        gen = generate_top5(item['question'], model)
        pred_set = set(gen['pred'])
        gold_set = set(canon(a) for a in item['articles'] if canon(a))

        tp = len(pred_set & gold_set)
        p = tp / max(len(pred_set), 1)
        r = tp / max(len(gold_set), 1)
        f1 = 2 * p * r / (p + r) if p + r > 0 else 0.0

        tp_total += tp
        pred_total += len(pred_set)
        gt_total += len(gold_set)

        results.append({
            'id': item['id'],
            'question': item['question'],
            'gold': list(gold_set),
            'pred': gen['pred'],
            'P': round(p, 6),
            'R': round(r, 6),
            'F1': round(f1, 6),
            'raw': gen['raw']
        })

    macro_p = sum(r['P'] for r in results) / len(results)
    macro_r = sum(r['R'] for r in results) / len(results)
    macro_f1 = sum(r['F1'] for r in results) / len(results)

    del model, base_model
    gc.collect()
    torch.cuda.empty_cache()

    row = {
        'label': label,
        'step': step,
        'P@5': round(macro_p, 4),
        'R@5': round(macro_r, 4),
        'F1@5': round(macro_f1, 4),
        'samples': len(results),
        'results': results
    }
    all_rows.append(row)
    print(f'  P@5={row["P@5"]:.4f}  R@5={row["R@5"]:.4f}  F1@5={row["F1@5"]:.4f}')

    with open(os.path.join(RESULTS_DIR, f'{label}.json'), 'w', encoding='utf-8') as f:
        json.dump(row, f, ensure_ascii=False, indent=2)

# ===== Leaderboard =====
print(f'\n{"="*60}')
print('FINAL LEADERBOARD')
print(f'{"="*60}')

rows = [{'Step': r['step'], 'Checkpoint': r['label'], 'P@5': r['P@5'], 'R@5': r['R@5'], 'F1@5': r['F1@5']} for r in all_rows]
df = pd.DataFrame(rows).sort_values('F1@5', ascending=False).reset_index(drop=True)
print(df.to_string(index=False))

best = df.iloc[0]
print(f'\n*** BEST: {best["Checkpoint"]}  F1@5={best["F1@5"]:.4f} ***')

with open(os.path.join(RESULTS_DIR, 'leaderboard.json'), 'w', encoding='utf-8') as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)
print(f'\nAll results saved to: {RESULTS_DIR}/')